# LOG-BASED BUG PATTERN MINING

## IMPORTING THE LIBRARIES

In [24]:
import numpy as np
import pandas as pd

## Loading the Dataset

In [25]:
df_log_based_bug_pattern_mining_dataset = pd.read_csv('synthetic_log_dataset (1).xls')

## Basic Dataset Overview

In [26]:
df_log_based_bug_pattern_mining_dataset.shape

(12500, 9)

In [27]:
df_log_based_bug_pattern_mining_dataset.head()

,log_id,timestamp,service_name,log_level,log_message,error_type,stack_trace,build_id,status
0,1,2025-09-01 00:00:00.000000,data_processor,DEBUG,Cache refreshed for user data,NoError,NaN,build-710-830,SUCCESS
1,2,2025-09-01 00:00:01.119059,payment_api,ERROR,NullPointerException: Required file missing on...,NullPointerException,NullPointerException occurred\n at payment_...,build-763-800,FAILURE
2,3,2025-09-01 00:00:02.216480,data_processor,INFO,User authentication completed,NoError,NaN,build-756-259,SUCCESS
3,4,2025-09-01 00:00:03.557185,analytics_service,INFO,Processed request successfully,NoError,NaN,build-357-230,SUCCESS
4,5,2025-09-01 00:00:05.350652,auth_service,WARN,None: Illegal argument passed to function,NoError,NaN,build-294-330,SUCCESS


In [28]:
df_log_based_bug_pattern_mining_dataset.tail()

,log_id,timestamp,service_name,log_level,log_message,error_type,stack_trace,build_id,status
12495,12496,2025-09-01 03:38:01.117914,auth_service,INFO,User authentication completed,NoError,NaN,build-885-131,SUCCESS
12496,12497,2025-09-01 03:38:02.779870,payment_api,ERROR,IllegalArgumentException: Required file missin...,IllegalArgumentException,IllegalArgumentException occurred\n at paym...,build-895-472,FAILURE
12497,12498,2025-09-01 03:38:04.176799,auth_service,WARN,None: Request timed out after waiting,NoError,NaN,build-980-111,SUCCESS
12498,12499,2025-09-01 03:38:05.801377,user_service,INFO,Query executed in 143 ms,NoError,NaN,build-337-807,SUCCESS
12499,12500,2025-09-01 03:38:07.081716,payment_api,INFO,Service heartbeat received,NoError,NaN,build-556-870,SUCCESS


In [29]:
df_log_based_bug_pattern_mining_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   log_id        12500 non-null  int64 
 1   timestamp     12500 non-null  object
 2   service_name  12500 non-null  object
 3   log_level     12500 non-null  object
 4   log_message   12500 non-null  object
 5   error_type    12500 non-null  object
 6   stack_trace   2479 non-null   object
 7   build_id      12500 non-null  object
 8   status        12500 non-null  object
dtypes: int64(1), object(8)
memory usage: 879.0+ KB


**Interpretation**
- We have 12500 observation and 9 attributes
- We have 1 numerical columns and 8 categorical columns
- Memory used is 879.0+ KB

## Null value handling

In [30]:
df_log_based_bug_pattern_mining_dataset.isnull().sum()

,0
log_id,0
timestamp,0
service_name,0
log_level,0
log_message,0
error_type,0
stack_trace,10021
build_id,0
status,0


**Interpretation**
- We have 10021 null values in stack_trace column

In [31]:
# We'll replace the missing values in 'stack_trace' with an empty string.
# .fillna('') replaces all 'NaN' values with an empty text string,
# which makes it easier to combine the columns later.
df_log_based_bug_pattern_mining_dataset['stack_trace'] = df_log_based_bug_pattern_mining_dataset['stack_trace'].fillna('')

In [32]:
df_log_based_bug_pattern_mining_dataset.isnull().sum()

,0
log_id,0
timestamp,0
service_name,0
log_level,0
log_message,0
error_type,0
stack_trace,0
build_id,0
status,0


**Interpretation**
- After replacing , we have 0 null values in the data set

In [33]:
print("\nDistribution of Error Types")
print(df_log_based_bug_pattern_mining_dataset['error_type'].value_counts())


Distribution of Error Types
error_type
NoError                      10021
NullPointerException           990
IndexOutOfBoundsException      401
FileNotFoundException          377
IllegalArgumentException       256
TimeoutException               233
ConnectionResetException       222
Name: count, dtype: int64


In [34]:
# We'll create a new column called 'full_text' by combining 'log_message' and 'stack_trace'.
# This new column contains all the text information related to each log entry.
df_log_based_bug_pattern_mining_dataset['full_text'] = df_log_based_bug_pattern_mining_dataset['log_message'] + ' ' + df_log_based_bug_pattern_mining_dataset['stack_trace']

In [35]:
# Convert all text in the 'full_text' column to lowercase.
# This is important for consistency, so a model treats 'NullPointerException' and 'nullpointerexception' the same.
df_log_based_bug_pattern_mining_dataset['full_text'] = df_log_based_bug_pattern_mining_dataset['full_text'].str.lower()

In [36]:
# Drop the original text columns from our DataFrame to keep it clean.
# 'axis=1' specifies that we are dropping a column, not a row.
df_log_based_bug_pattern_mining_dataset = df_log_based_bug_pattern_mining_dataset.drop(columns=['log_message', 'stack_trace'])

In [37]:
# Display the prepared DataFrame to see the changes.
print("\n Prepared DataFrame (after cleaning) ")
df_log_based_bug_pattern_mining_dataset.head()


 Prepared DataFrame (after cleaning) 


,log_id,timestamp,service_name,log_level,error_type,build_id,status,full_text
0,1,2025-09-01 00:00:00.000000,data_processor,DEBUG,NoError,build-710-830,SUCCESS,cache refreshed for user data
1,2,2025-09-01 00:00:01.119059,payment_api,ERROR,NullPointerException,build-763-800,FAILURE,nullpointerexception: required file missing on...
2,3,2025-09-01 00:00:02.216480,data_processor,INFO,NoError,build-756-259,SUCCESS,user authentication completed
3,4,2025-09-01 00:00:03.557185,analytics_service,INFO,NoError,build-357-230,SUCCESS,processed request successfully
4,5,2025-09-01 00:00:05.350652,auth_service,WARN,NoError,build-294-330,SUCCESS,none: illegal argument passed to function


In [38]:
print("\n Final DataFrame Info ")
df_log_based_bug_pattern_mining_dataset.info()


 Final DataFrame Info 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12500 entries, 0 to 12499
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   log_id        12500 non-null  int64 
 1   timestamp     12500 non-null  object
 2   service_name  12500 non-null  object
 3   log_level     12500 non-null  object
 4   error_type    12500 non-null  object
 5   build_id      12500 non-null  object
 6   status        12500 non-null  object
 7   full_text     12500 non-null  object
dtypes: int64(1), object(7)
memory usage: 781.4+ KB


## Feature Engineering (TF-IDF)

In [39]:
# Import the TfidfVectorizer from scikit-learn.
# 'scikit-learn' is the most popular library for machine learning in Python.
from sklearn.feature_extraction.text import TfidfVectorizer

In [40]:
# Initialize the TF-IDF Vectorizer.
# 'TfidfVectorizer' is a tool that does all the TF-IDF calculations for you.
# We set a few parameters to improve performance:
# - max_features=5000: This tells the vectorizer to only consider the 5000 most frequent words.
#   This helps reduce the size of our dataset and focuses on the most important terms.
# - stop_words='english': This automatically removes common English words like 'the', 'a', 'is', etc.
#   These words don't carry much meaning for bug pattern mining, so we can ignore them.
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

In [41]:
# Now, we use the vectorizer to transform our text data.
# 'fit_transform()' does two things in one step:
# 1. 'fit': It learns the vocabulary (all the unique words) from our data.
# 2. 'transform': It converts each log entry into a numerical vector based on the TF-IDF scores.
# The result is stored in a variable called 'X'.
X = tfidf_vectorizer.fit_transform(df_log_based_bug_pattern_mining_dataset['full_text'])

In [42]:
# We also need to define our target variable, which is what we want to predict.
# Our target is the 'error_type' column, and we'll store it in a variable called 'y'.
y = df_log_based_bug_pattern_mining_dataset['error_type']

In [43]:
print("\nFeature Matrix Shape ")
# This prints the size of our numerical dataset.
# The first number is the number of log entries (rows), and the second is the number of features (words).
# For example, (12500, 5000) means we have 12,500 logs and 5,000 features.
print(f"Shape of X (features): {X.shape}")
print(f"Shape of y (target): {y.shape}")


Feature Matrix Shape 
Shape of X (features): (12500, 546)
Shape of y (target): (12500,)


## Model Training and Evaluation

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [45]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.3, random_state=42, stratify=y)

In [46]:
# Initialize the Logistic Regression model.
# Logistic Regression is a simple but effective model for classification tasks.
# The 'class_weight='balanced'' parameter automatically adjusts for the class imbalance
# by giving more importance to the less frequent bug types.
model = LogisticRegression(class_weight='balanced', solver='liblinear')

In [47]:
# Train the model.
# The '.fit()' method "trains" the model by learning the relationship between the features (X_train)
# and the target labels (y_train).
print("Training the model")
model.fit(X_train, y_train)
print("Model training complete.")

Training the model
Model training complete.


In [48]:
# Model Evaluation
# Make predictions on the test set.
# The '.predict()' method uses the trained model to predict the error type for new, unseen data (X_test).
y_pred = model.predict(X_test)

In [49]:
# Print the classification report.
# This report gives you a detailed look at your model's performance for each bug type.
print("\nClassification Report")
print(classification_report(y_test, y_pred, zero_division=0))


Classification Report
                           precision    recall  f1-score   support

 ConnectionResetException       1.00      1.00      1.00        67
    FileNotFoundException       1.00      1.00      1.00       113
 IllegalArgumentException       1.00      1.00      1.00        77
IndexOutOfBoundsException       1.00      1.00      1.00       120
                  NoError       1.00      1.00      1.00      3006
     NullPointerException       1.00      1.00      1.00       297
         TimeoutException       1.00      1.00      1.00        70

                 accuracy                           1.00      3750
                macro avg       1.00      1.00      1.00      3750
             weighted avg       1.00      1.00      1.00      3750



In [50]:
# Print the confusion matrix.
# This matrix shows you where your model was right and where it was wrong.
print("\nConfusion Matrix")
cm = confusion_matrix(y_test, y_pred)
# We'll use pandas to make the confusion matrix more readable with row and column labels.
cm_df = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
print(cm_df)


Confusion Matrix
                           ConnectionResetException  FileNotFoundException  \
ConnectionResetException                         67                      0   
FileNotFoundException                             0                    113   
IllegalArgumentException                          0                      0   
IndexOutOfBoundsException                         0                      0   
NoError                                           0                      0   
NullPointerException                              0                      0   
TimeoutException                                  0                      0   

                           IllegalArgumentException  \
ConnectionResetException                          0   
FileNotFoundException                             0   
IllegalArgumentException                         77   
IndexOutOfBoundsException                         0   
NoError                                           0   
NullPointerException       

In [51]:
X_text = df_log_based_bug_pattern_mining_dataset['full_text']
y = df_log_based_bug_pattern_mining_dataset['error_type']


In [52]:
from sklearn.pipeline import Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('classifier', LogisticRegression(solver='liblinear'))
])

In [53]:
param_grid = {
    'tfidf__max_features': [2000, 5000, 10000],
    'classifier__C': [0.1, 1, 10],
    'classifier__class_weight': ['balanced', None]
}

In [54]:
from sklearn.model_selection import GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)
print("Starting Grid Search for optimal parameters")
grid_search.fit(X_text, y)
print("Grid Search complete.")

Starting Grid Search for optimal parameters
Grid Search complete.


In [55]:
# Print the best parameters and the best score
print("\nBest Parameters Found")
print(grid_search.best_params_)
print("\nBest F1-Score")
print(grid_search.best_score_)


Best Parameters Found
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'tfidf__max_features': 2000}

Best F1-Score
1.0
